## ⚠️ Data Confidentiality Notice

The data contained in this notebook is **proprietary easyJet data** and is strictly confidential.

* **Do not** download, copy or export this data from the Databricks platform.
* **Do not** share, email or send this data to anyone, both internally or externally.
* **Do not** reproduce this data in any external tool, document or personal device.

All work must remain within this Databricks workspace. Any breach of these conditions may constitute a violation of easyJet's data protection policies.

If you are unsure whether a particular action is permitted, please ask a member of the team before proceeding.

In [0]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [0]:
df_raw = spark.sql("""
    select 
        flightnumber, 
        flight_date, 
        local_departure_time,
        local_arrival_time, 
        total_flight_time, 
        seatssold, 
        capacity,
        departure_iata, 
        arrival_iata,
        alpha_route, 
        route_type,
        departure_country, 
        arrival_country, 
        sum_leisure_reasonfortravel,
        sum_business_reasonfortravel, 
        sum_other_reasonfortravel,
        sum_mobile_app_commercialchannellevel2,
        sum_web_commercialchannellevel2, 
        sum_other_commercialchannellevel2,
        sum_standard_bookingtickettype, 
        sum_staff_promo_bookingtickettype,
        sum_inclusive_bookingtickettype, 
        sum_other_bookingtickettype,
        avg_ticket_price, 
        min_ticket_price, 
        max_ticket_price,
        avg_lead_time, 
        min_lead_time, 
        max_lead_time,
        avg_seat_per_reservation, 
        min_seat_per_reservation,
        max_seat_per_reservation
    from {flight_table};
    """).toPandas()

df_raw.head(10)

## Task overview:
- The below questions are designed to guide you to explore and understand the data further.
- In your approach you can use, amend, ignore or create your own questions.
- Try to leverage the full power of the dataset by understanding each of the features and using them to inform your analysis. 
- You are encouraged utilise the existing features as well as extract/combine/enhance to create "new" features. 
- Many of the patterns you will see are intuitive, as an Analyst it is always good to think of the "Why", this will make the analysis more engaging and enjoyable.
- While it is tempting to use AI to generate a solution for you, I would encourage you to avoid doing so. Recall the coding knowledge you already have, reasearch via reading documentation/Stack Overflow and ask the team around you for tips. We have a very supportive team that will be willing to take time out of their day to answer questions and explain concepts that are unfamiliar to you.
This approach will help you gain the most out of these two weeks.
Have fun! 

## Dataset Description

This dataset contains operational and commercial attributes for easyJet flights. Each row represents a single flight and includes:

* **Flight details** - flight number, date, local departure/local arrival times and flight duration
* **Capacity & demand** - seats sold and aircraft capacity 
* **Route information** - departure/arrival airports, countries and route type (Beach, City etc)
* **Passenger booking behaviour** -  reason for travel (leisure, business, other), booking method (mobile app, web, other) and ticket type (standard, staff/promo, inclusive, other)
* **Pricing** - average, minimum, and maximum ticket price per flight
* **Lead time** - average, minimum, and maximum days between booking date and departure date
* **Booking reservation** - average, minimum and maximum seats per reservation

The goal is to explore what drives load factor and ultimately predict load factor at the time of departure.

### 1. Understanding the Dataset
- How many rows and columns does the dataset have?
- What data types are used to store the data in each column? Are there any that look incorrect (eg: numbers stored as text)?
- Are there any missing values? Which columns are impacted and do you need to apply a correction?

**TIP**: use `.describe()` and `.info()` for basic stats.
Python stores items as objects in memory, you can then apply different commands to these objects.

In [0]:
# Amend this to display summary information about the DataFrame. 
# Explore other commands and see which are useful in your analysis

### Code ###




In [0]:
df = pd.DataFrame(df_raw)

print("Columns = ", df.shape[1])
print ("Rows = ", df.shape[0])



In [0]:
print("Old Dataset") 
print(df.dtypes)
df1 = df.copy()

df1['flightnumber'] =pd.to_numeric(df['flightnumber'])
df1['flight_date'] = pd.to_datetime(df['flight_date'])
df1['avg_ticket_price'] = pd.to_numeric(df['avg_ticket_price'])
df1['min_ticket_price'] = pd.to_numeric(df['min_ticket_price'])
df1['max_ticket_price'] = pd.to_numeric(df['max_ticket_price'])

print("New Dataset") 
print(df1.dtypes)

In [0]:
print(df1.isnull().sum())




In [0]:


missing_ds=df1.loc[df1.route_type.isnull()]
i = 0
saved = 0
loop = missing_ds.shape[0]
arr_iata = missing_ds.arrival_iata.unique()  #the arrival airports of only missing values
while i < len(arr_iata):
   
    mode_result = df1.loc[df1['arrival_iata'] == arr_iata[0+i],'route_type'].mode()
    if len(mode_result) == 0:
        df1=df1.loc[~(df1['route_type'].isnull() & (df1['arrival_iata'] == arr_iata[0+i]))]
        i = i+1
        continue
    common = mode_result[0]
    common1 = df1.loc[df1['arrival_iata'] == arr_iata[0+i],'route_type'].value_counts(normalize=True).max() *100


        
    if common1 > 75:
        df1.loc[(df1['route_type'].isnull()) & (df1['arrival_iata'] == arr_iata[0+i]),'route_type'] = common
        saved = saved +1
    else:
        df1=df1.loc[~(df1['route_type'].isnull() & (df1['arrival_iata'] == arr_iata[0+i]))]

    i = i+1
    

print(saved)   
df1     

In [0]:
oob_iata = [iata for iata in df.arrival_iata.unique() if iata not in df1.arrival_iata.unique()]

print(oob_iata)
print("Nr of oob iata dropped: ", len(oob_iata))


### 2. Descriptive Statistics
- What is the average, minimum and maximum "load_factor" across all flights?
- What is the average ticket price and how does it vary?
- What is the typical length of a flight? What are the longest and shortest flights that easyJet fly?

In [0]:
# The Python language has a large user base and is very well documented. Use this to your advantage when trying to find out how to do things.
# From the below link, can you work out how to find the minimum value of a column in a dataset?
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.min.html

### Code ###

df1['load_factor'] = (df1['seatssold'] / df1['capacity']) * 100


print("This is the Min Load Factor ", df1['load_factor'].min(),"This is the Max Load Factor ", df1['load_factor'].max(), "This is the Average Load Factor", df1['load_factor'].mean())

print("This is the Average Ticket Price ", df1['avg_ticket_price'].mean())

print("This is the Shortest Flight Time = ", df1['total_flight_time'].min()," This is the Longest Flight Time = ", df1['total_flight_time'].max(), " This is the Typical Flight Time = ", df1['total_flight_time'].mean())


In [0]:
df1['flightnumber'] =pd.to_numeric(df['flightnumber'])
df1.loc[df1['total_flight_time'] == 1530]

### 3. Routes & Geography
- How many unique routes (`alpha_route`) are in the dataset?
- Which departure airport has the most flights?
- What are the top 5 most common destination countries?
- What types of routes do easyJet fly?

Many of the questions you have about your dataset are likely to have been answered by someone else on their own dataset. Use this to your advantage when you are trying to find out how to do things.

Stack Overflow has real questions and real answers from developers across the world. This is different from asking an AI Agent to solve something for you. Here, you will find multiple approaches to the same question, you will discover the reason for why people may have preferred one approach over another and you will develop your own unique coding style and preferences. 

https://stackoverflow.com/questions

In [0]:
# Below is a link to a StackOverflow discussion on how to find the minimum value of a column in a dataset.
# Exploring different ways of doing the same thing will help you become a better problem solver when dealing with data.

# https://stackoverflow.com/questions/47006617/finding-max-min-value-of-individual-columns

# Leverage the Python community to your advantage!

### Code ###

alpha_unique = df1.alpha_route.unique()
print(alpha_unique)
print("The total number of unique routes is",len(alpha_unique))

com_dep = df1['departure_iata'].mode()[0]
print ("The most common Departure Airport is", com_dep)

com_des = df1["arrival_country"].value_counts().head(5)
print("Top 5 Most Common Destination Countries: ")
print(com_des)

types = df1.route_type.unique()
print("The types of routes are",types)




### 4. Booking Behaviour
- What proportion of passengers book for leisure/business/other reasons?
- Which booking channel is most popular?
- What is the most common ticket type?

**TIP**: use `.value_counts()` to find the number of unique values in a column

In [0]:
# Python can be used as a calculator.
# Try execute the below:
print(1+1)

# Typing individual numbers in can be a pain and messy, you can also do the below:
print(sum([1,2,3,4,5]))

# Again, sometimes the numbers may be the result of another aggregation. The below is also valid:
# eg: I bought 5 apples and 7 oranges, how many fruits did I buy?
apples = 5
oranges = 7
print(apples + oranges)

In [0]:
### Code ###

leisure = (df1['sum_leisure_reasonfortravel'] / df1['seatssold']) *100
business = (df1['sum_business_reasonfortravel'] / df1['seatssold']) *100
other = (df1['sum_other_reasonfortravel'] / df1['seatssold']) *100
ss = df1['seatssold']
unknown = (100 - leisure.mean() - business.mean() - other.mean())
print("The percentage of leisure passengers is ", leisure.mean(), "The percentage of business passengers is ", business.mean(), "The percentage of other passengers is ", other.mean(), "The percentage of unknown passengers is ", unknown.mean())


df1["percentage_leisure"] = df1.apply(lambda x: (x["sum_leisure_reasonfortravel"]/x["seatssold"])*100, axis=1)
df1["percentage_business"] = df1.apply(lambda x: (x["sum_business_reasonfortravel"]/x["seatssold"])*100, axis=1)
df1["percentage_other"] = df1.apply(lambda x: (x["sum_other_reasonfortravel"]/x["seatssold"])*100, axis=1)

df1["percentage_mobile"] = df1.apply(lambda x: (x["sum_mobile_app_commercialchannellevel2"]/x["seatssold"])*100, axis=1)
df1["percentage_web"] = df1.apply(lambda x: (x["sum_web_commercialchannellevel2"]/x["seatssold"])*100, axis=1)
df1["percentage_otherchannel"] = df1.apply(lambda x: (x["sum_other_commercialchannellevel2"]/x["seatssold"])*100, axis=1)


df1["percentage_staff"] = df1.apply(lambda x: (x["sum_staff_promo_bookingtickettype"]/x["seatssold"])*100, axis=1)
df1["percentage_inclusive"] = df1.apply(lambda x: (x["sum_inclusive_bookingtickettype"]/x["seatssold"])*100, axis=1)
df1["percentage_standard"] = df1.apply(lambda x: (x["sum_standard_bookingtickettype"]/x["seatssold"])*100, axis=1)
df1["perecentage_otherticket"] = df1.apply(lambda x: (x["sum_other_bookingtickettype"]/x["seatssold"])*100, axis=1)






mobile = (df1['sum_mobile_app_commercialchannellevel2'] / df1['seatssold']) *100
web = (df1['sum_web_commercialchannellevel2'] / df1['seatssold']) *100
other1 = (df1['sum_other_commercialchannellevel2'] / df1['seatssold']) *100
unknown1 = (100 - mobile.mean() - web.mean() - other1.mean())

if mobile.mean() > web.mean() and mobile.mean() > other1.mean() and mobile.mean() > unknown1.mean():
    print("Mobile is the most popular channel")
    print ("Total Mobile = ,", mobile.mean())
    print ("Total Web = ", web.mean())
    print ("Total Other = ", other1.mean())
    print ("Total Unknown = ", unknown1.mean())

elif web.mean() > mobile.mean() and web.mean() > other1.mean() and web.mean() > unknown1.mean():
    print("Web is the most popular channel")
    print ("Total Web = ", web.mean())
    print ("Total Mobile = ,", mobile.mean())
    print ("Total Other = ", other1.mean())
    print ("Total Unknown = ", unknown1.mean())

elif other1.mean() > mobile.mean() and other1.mean() > web.mean() and other1.mean() > unknown1.mean():
    print("Other is the most popular channel")
    print ("Total Other = ", other1.mean())
    print ("Total Mobile = ,", mobile.mean())
    print ("Total Web = ", web.mean())
    print ("Total Unknown = ", unknown1.mean())

elif unknown1.mean() > mobile.mean() and unknown1.mean() > web.mean() and unknown1.mean() > other1.mean():
    print("Unknown is the most popular channel")
    print ("Total Unknown = ", unknown1.mean())
    print ("Total Other = ", other1.mean())
    print ("Total Mobile = ,", mobile.mean())
    print ("Total Web = ", web.mean())
    

inclusive = (df1['sum_inclusive_bookingtickettype'] / df1['seatssold']) *100
standard = (df1['sum_standard_bookingtickettype'] / df1['seatssold']) *100
staff = (df1['sum_staff_promo_bookingtickettype'] / df1['seatssold']) *100
other2 = (df1['sum_other_bookingtickettype'] / df1['seatssold']) *100
unknown2 = (100 - inclusive.mean() - standard.mean() - staff.mean() - other2.mean())

if inclusive.mean() > standard.mean() and inclusive.mean() > staff.mean() and inclusive.mean() > other2.mean() and inclusive.mean() > unknown2.mean():
    print("Inclusive is the most common booking type")
    print ("Total Inclusive = ", inclusive.mean())
    print ("Total Standard = ", standard.mean())
    print ("Total Staff = ", staff.mean())
    print ("Total Other = ", other2.mean())
    print ("Total Unknown = ", unknown2.mean())

elif standard.mean() > inclusive.mean() and standard.mean() > staff.mean() and standard.mean() > other2.mean() and standard.mean() > unknown2.mean():
    print("Standard is the most common booking type")
    print ("Total Standard = ", standard.mean())
    print ("Total Inclusive = ", inclusive.mean())
    print ("Total Staff = ", staff.mean())
    print ("Total Other = ", other2.mean())
    print ("Total Unknown = ", unknown2.mean())

elif staff.mean() > inclusive.mean() and staff.mean() > standard.mean() and staff.mean() > other2.mean() and staff.mean() > unknown2.mean():
    print("Staff is the most common booking type")
    print ("Total Staff = ", staff.mean())
    print ("Total Inclusive = ", inclusive.mean())
    print ("Total Standard = ", standard.mean())
    print ("Total Other = ", other2.mean())
    print ("Total Unknown = ", unknown2.mean())

elif other2.mean() > inclusive.mean() and other2.mean() > standard.mean() and other2.mean() > staff.mean() and other2.mean() > unknown2.mean():
    print("Other is the most common booking type")
    print ("Total Other = ", other2.mean())
    print ("Total Inclusive = ", inclusive.mean())
    print ("Total Standard = ", standard.mean())
    print ("Total Staff = ", staff.mean())
    print ("Total Unknown = ", unknown2.mean())

elif unknown2.mean() > inclusive.mean() and unknown2.mean() > standard.mean() and unknown2.mean() > staff.mean() and unknown2.mean() > other2.mean():
    print("Unknown is the most common booking type")
    print ("Total Unknown = ", unknown2.mean())
    print ("Total Other = ", other2.mean())
    print ("Total Inclusive = ", inclusive.mean())
    print ("Total Standard = ", standard.mean())
    print ("Total Staff = ", staff.mean())

df1

In [0]:
unknown = (ss.mean() - (leisure.mean() + business.mean() + other.mean()))
print (unknown)
print (ss.mean())
print ((leisure.mean() + business.mean() + other.mean()))

### 5. Load Factor Analysis
- How is load factor distributed across easyjet flights? (Hint: try plot a histogram)
- Which route types tend to have the highest average load factor?
- Is there a relationship between `avg_ticket_price` and `load_factor`? (Hint: scatter plot)

**TIP**: There are some libraries that can be installed to create advanced plots such as `matplotlib` and `seaborn`, but you can also create basic plots directly from the pandas dataframe itself using `df.plot()`

Enhance this further using the pandas docsctring: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html

In [0]:
%skip
df1['load_factor'].plot(kind='hist', color='blue', alpha=0.5, figsize=(15,5))

In [0]:
%skip
### Code ###   



df1['load_factor'].hist(bins=90, color='blue', alpha=0.7, figsize=(15,5))











In [0]:
%skip


df1.groupby('route_type').mean()['load_factor'].plot(kind='bar', color='blue', alpha=0.7, figsize=(15,5))


In [0]:
load = df1['load_factor'].mean()
print(load)

In [0]:
print (df1.groupby('route_type').count()['load_factor'])

In [0]:
df1.plot(kind='scatter', x='load_factor', y='avg_ticket_price', color='blue', alpha=0.7, figsize=(15,5))

### 6. Time Based Patterns
- Which month has the fewest/most flights?
- Does average load factor change depending on whether a flight departs in the morning vs evening?
- How do ticket prices change across a year, do you notice any seasonality trends? 

In [0]:
# What happens when you run the below? Can you leverage this "new" column, combine it with a groupby count and answer the first question?
date_column = 'flight_date'
df1['month'] = df1['flight_date'].dt.month


### Code ###

groupby_month = df1.groupby('month').count()['flightnumber']
print ("These Months had this many flights:")
print(groupby_month)

In [0]:
df1['hour'] = df1['local_departure_time'].dt.hour

morning = df1.loc[df1['hour'] < 12, 'load_factor']
evening = df1.loc[(df1['hour'] >= 17) & (df1['hour'] <= 21), 'load_factor']


print("The average load factor for morning flights is:")
print (morning.mean())
print("The average load factor for evening flights is:")
print (evening.mean())


In [0]:
spring = df1.loc[(df1['month'] >= 3) & (df1['month'] <= 5), 'avg_ticket_price']
summer = df1.loc[(df1['month'] >= 6) & (df1['month'] <= 8), 'avg_ticket_price']
autumn = df1.loc[(df1['month'] >= 9) & (df1['month'] <= 11), 'avg_ticket_price']
winter = df1.loc[(df1['month'] == 12) | (df1['month'] <= 2), 'avg_ticket_price']

print("The average ticket price for spring is:")
print (spring.mean())
print("The average ticket price for summer is:")
print (summer.mean())
print("The average ticket price for autumn is:")
print (autumn.mean())
print("The average ticket price for winter is:")
print (winter.mean())

In [0]:
df1.loc[df1['month'] == 12]

### 7. Machine Learning

**Context:** You are a data scientist at an airline. The management team have noticed that some flights consistently depart with empty seats resulting in lost revenue. They have asked you to work with the digital marketing department and try to identify these flights early enough to intervene. The aim is to try to fill these seats via various promotions for those routes.

**Your task:** Build a linear regression model that predicts the load factor of a flight so that easyJet can enhance their approach in sending out promotional content.

Flights predicted to have spare capacity become candidates for promotional campaigns, flights already predicted to fill naturally require no intervention.

**Guidance:**
- Begin by looking at features that are correlated with load factor
- Choosing just one of these features, how well can you build a model to predict load factor? (Hint: y=mx+c)
- How do you evaluate the "goodness" of the model built?
- Try incrementally adding more features, does the model improve or degrade as you add/remove certain features?
(y = m1x1 + m2x2 .. + c)
- Evaluate performance - How "good" do you feel your model is? (Explain in business terms, what does it mean and what are the consequences of over/under predicting load factor given your business problem?) 
- Evaluate the problem - Is the business goal realistic, is the data sufficient, what additional features would you have liked to have in order to better tackle this problem?

In [0]:
%skip
### Code ###
v = 0
column_names = list(df1.columns)

while v <33:
    df1.plot(kind='scatter', x='load_factor', y=column_names[v], color='blue', alpha=0.02, figsize=(15,5))
    v=v+1




In [0]:
%skip
df1.groupby('departure_country').mean()['load_factor'].sort_values(ascending=True).plot(kind='bar', color='blue', alpha=0.7, figsize=(15,5))



In [0]:
%skip
df1.groupby('arrival_country').mean()['load_factor'].sort_values(ascending=True).plot(kind='bar', color='blue', alpha=0.7, figsize=(15,5))

In [0]:
df1[['departurecountry', 'arrivalcountry', 'routetype']] = df1[['departure_country', 'arrival_country', 'route_type']].copy()
df1 = pd.get_dummies(df1, columns=['departure_country', 'arrival_country'])
df1 = pd.get_dummies(df1, columns=['route_type'])


In [0]:
df1.loc[df1['alpha_route'] == 'ALCLPL']

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

df1 = df1.sort_values('local_departure_time')

df1['log_avg_tp'] = np.log1p(df1['avg_ticket_price'])
df1['log_min_tp'] = np.log1p(df1['min_ticket_price'])
df1['log_max_tp'] = np.log1p(df1['max_ticket_price'])
df1['total_fly_time'] = np.log1p(df1['total_flight_time'])
df1['standard'] = np.log1p(df1['sum_standard_bookingtickettype'])
df1['staff'] = np.log1p(df1['sum_staff_promo_bookingtickettype'])
df1['inclusive'] = np.log1p(df1['sum_inclusive_bookingtickettype'])
df1['other'] = np.log1p(df1['sum_other_bookingtickettype'])
df1['web_channel'] = np.log1p(df1['sum_web_commercialchannellevel2'])
df1['mobile_channel'] = np.log1p(df1['sum_mobile_app_commercialchannellevel2'])
df1['other_channel'] = np.log1p(df1['sum_other_commercialchannellevel2'])




test_columns = ['log_avg_tp', 'departure_country_Belgium', 'departure_country_Bulgaria', 'departure_country_Croatia', 'departure_country_Cyprus', 'departure_country_Czechia', 'departure_country_Denmark', 'departure_country_Finland', 'departure_country_France', 'departure_country_Germany', 'departure_country_Greece', 'departure_country_Hungary', 'departure_country_Iceland', 'departure_country_Italy', 'departure_country_Luxembourg', 'departure_country_Malta', 'departure_country_Netherlands', 'departure_country_Norway', 'departure_country_Poland', 'departure_country_Portugal', 'departure_country_Other', 'departure_country_Slovenia', 'departure_country_Spain', 'departure_country_Sweden', 'departure_country_Switzerland', 'departure_country_United Kingdom', 'arrival_country_Belgium', 'arrival_country_Bulgaria', 'arrival_country_Croatia', 'arrival_country_Cyprus', 'arrival_country_Czechia', 'arrival_country_Denmark', 'arrival_country_Finland', 'arrival_country_France', 'arrival_country_Germany', 'arrival_country_Greece', 'arrival_country_Hungary', 'arrival_country_Iceland', 'arrival_country_Italy','arrival_country_Luxembourg', 'arrival_country_Malta', 'arrival_country_Netherlands', 'arrival_country_Norway', 'arrival_country_Poland', 'arrival_country_Portugal', 'arrival_country_Other', 'arrival_country_Slovenia', 'arrival_country_Spain', 'arrival_country_Sweden', 'arrival_country_Switzerland', 'arrival_country_United Kingdom', 'route_type_Beach','route_type_City','route_type_Domestic','route_type_Non EUR', 'log_min_tp', 'log_max_tp','avg_lead_time','total_fly_time','standard','staff','inclusive','other','web_channel','mobile_channel','other_channel','capacity']

target = 'load_factor'


train1, test1, train2 , test2 = train_test_split (df1[test_columns], df1[target], test_size=0.2, shuffle= False)




In [0]:
df1['past_seatssold'] = (df1['sum_business_reasonfortravel'] + df1['sum_leisure_reasonfortravel'] + df1['sum_other_reasonfortravel'])
df1['past_loadfactor'] = ((df1['past_seatssold']/df1['capacity']) * 100)


df1['alphalf'] = (                                                               # past data for load factor based on route
    df1.groupby('alpha_route')['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['alphalf'] = df1['alphalf'].fillna(df1['load_factor'].mean())

df1['alpha_lag_lf'] = (                                                              
    df1.groupby('alpha_route')['load_factor']
       .transform(lambda x: x.rolling(window=3).mean().shift(1))
)
df1['alpha_lag_lf'] = df1['alpha_lag_lf'].fillna(df1['load_factor'].mean())

conditions = [
    df1['month'].isin([1]),
    df1['month'].isin([2]),
    df1['month'].isin([3]),
    df1['month'].isin([4]),
    df1['month'].isin([5]),
    df1['month'].isin([6]),
    df1['month'].isin([7]),
    df1['month'].isin([8]),
    df1['month'].isin([9]),
    df1['month'].isin([10]),
    df1['month'].isin([11]),
    df1['month'].isin([12])
]
choices = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
                                                                                       
df1['Months'] = np.select(conditions, choices)# past data based on seasons

df1['monthlf'] = (
    df1.groupby('Months')['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['monthlf'] = df1['monthlf'].fillna(df1['load_factor'].mean())

df1['alphabymonthlf'] = (
    df1.groupby(['Months','alpha_route'])['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['alphabymonthlf'] = df1['alphabymonthlf'].fillna(df1['load_factor'].mean())

df1['dayoftheweek'] = df1['flight_date'].dt.dayofweek

df1['alphabydaylf'] = (
    df1.groupby(['dayoftheweek','alpha_route'])['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['alphabydaylf'] = df1['alphabydaylf'].fillna(df1['load_factor'].mean())

df1['dayoftheweek'] = df1['flight_date'].dt.dayofweek

df1['daylf'] = (
    df1.groupby('dayoftheweek')['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['daylf'] = df1['daylf'].fillna(df1['load_factor'].mean())








test_columns = ['log_avg_tp', 'departure_country_Belgium', 'departure_country_Bulgaria', 'departure_country_Croatia', 'departure_country_Cyprus', 'departure_country_Czechia', 'departure_country_Denmark', 'departure_country_Finland', 'departure_country_France', 'departure_country_Germany', 'departure_country_Greece', 'departure_country_Hungary', 'departure_country_Iceland', 'departure_country_Italy', 'departure_country_Luxembourg', 'departure_country_Malta', 'departure_country_Netherlands', 'departure_country_Norway', 'departure_country_Poland', 'departure_country_Portugal', 'departure_country_Other', 'departure_country_Slovenia', 'departure_country_Spain', 'departure_country_Sweden', 'departure_country_Switzerland', 'departure_country_United Kingdom', 'arrival_country_Belgium', 'arrival_country_Bulgaria', 'arrival_country_Croatia', 'arrival_country_Cyprus', 'arrival_country_Czechia', 'arrival_country_Denmark', 'arrival_country_Finland', 'arrival_country_France', 'arrival_country_Germany', 'arrival_country_Greece', 'arrival_country_Hungary', 'arrival_country_Iceland', 'arrival_country_Italy','arrival_country_Luxembourg', 'arrival_country_Malta', 'arrival_country_Netherlands', 'arrival_country_Norway', 'arrival_country_Poland', 'arrival_country_Portugal', 'arrival_country_Other', 'arrival_country_Slovenia', 'arrival_country_Spain', 'arrival_country_Sweden', 'arrival_country_Switzerland', 'arrival_country_United Kingdom', 'route_type_Beach','route_type_City','route_type_Domestic','route_type_Non EUR', 'log_min_tp', 'log_max_tp','avg_lead_time','total_fly_time','standard','staff','inclusive','other','web_channel','mobile_channel','other_channel','alphalf','monthlf','past_loadfactor','capacity','daylf','alphabymonthlf','alphabydaylf','alpha_lag_lf']

train1, test1, train2 , test2 = train_test_split (df1[test_columns], df1[target], test_size=0.2, shuffle = False)


In [0]:
lfmodel = LinearRegression()

lfmodel.fit(train1, train2)

prediction = lfmodel.predict(test1)





score = r2_score(test2, prediction)
mae = mean_absolute_error(test2, prediction)

print("The score is: ", score)  # more higher = you know more
print("Mean squared error:" , mean_squared_error(test2, prediction)) # lower is better
print("Mean absolute error is ", mae)  # 0.0 is best



In [0]:
df1

In [0]:
%skip
print(df1['standard'].corr(df1['past_seatssold']))
print(df1['inclusive'].corr(df1['past_seatssold']))
print(df1['other'].corr(df1['past_seatssold']))
print(df1['staff'].corr(df1['past_seatssold']))
print(df1['web_channel'].corr(df1['past_seatssold']))
print(df1['mobile_channel'].corr(df1['past_seatssold']))
print(df1['other_channel'].corr(df1['past_seatssold']))
print(df1['log_avg_tp'].corr(df1['past_seatssold']))
print(df1['log_min_tp'].corr(df1['past_seatssold']))
print(df1['log_max_tp'].corr(df1['past_seatssold']))
print(df1['avg_lead_time'].corr(df1['past_seatssold']))
print(df1['total_fly_time'].corr(df1['past_seatssold']))
print(df1['standard'].corr(df1['web_channel']))
print(df1['inclusive'].corr(df1['web_channel']))
print(df1['other'].corr(df1['web_channel']))
print(df1['staff'].corr(df1['web_channel']))
print('1', df1['standard'].corr(df1['mobile_channel']))
print(df1['inclusive'].corr(df1['mobile_channel']))
print(df1['other'].corr(df1['mobile_channel']))
print(df1['staff'].corr(df1['mobile_channel']))
print(df1['standard'].corr(df1['other_channel']))
print(df1['inclusive'].corr(df1['other_channel']))
print(df1['other'].corr(df1['other_channel']))
print(df1['staff'].corr(df1['other_channel']))




In [0]:
%skip

df1.plot(kind='scatter', x='load_factor', y='inclusive', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='other', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='standard', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='staff', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='other_channel', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='web_channel', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='mobile_channel', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='past_seatssold', color='blue', alpha=0.7, figsize=(15,5))




df1.plot(kind='scatter', x='load_factor', y='avg_seat_res', color='blue', alpha=0.7, figsize=(15,5))
df1.plot(kind='scatter', x='load_factor', y='max_seat_res', color='blue', alpha=0.7, figsize=(15,5))


In [0]:
%pip install scikit-optimize
%pip install LightGBM
%pip install xgboost

In [0]:
import numpy as np
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df1 = df1.sort_values('local_departure_time')

df1['departurecountry'] = df1['departurecountry'].astype('category')
df1['arrivalcountry'] = df1['arrivalcountry'].astype('category')
df1['routetype'] = df1['routetype'].astype('category')


df1['sum_standard'] = (df1['sum_standard_bookingtickettype']/df1['capacity'])*100
df1['sum_staff'] = (df1['sum_staff_promo_bookingtickettype']/df1['capacity'])*100
df1['sum_inclusive'] = (df1['sum_inclusive_bookingtickettype']/df1['capacity'])*100
df1['sum_other'] = (df1['sum_other_bookingtickettype']/df1['capacity'])*100
df1['sum_web_c'] = (df1['sum_web_commercialchannellevel2']/df1['capacity'])*100
df1['sum_mobile_app_c'] = (df1['sum_mobile_app_commercialchannellevel2']/df1['capacity'])*100
df1['sum_other_c'] = (df1['sum_other_commercialchannellevel2']/df1['capacity'])*100
df1['avg_seat_res'] = (df1['avg_seat_per_reservation']/df1['capacity'])*100
df1['max_seat_res'] = (df1['max_seat_per_reservation']/df1['capacity'])*100
df1['min_seat_res'] = (df1['min_seat_per_reservation']/df1['capacity'])*100






X = df1[['avg_ticket_price','min_ticket_price', 'max_ticket_price','avg_lead_time','total_flight_time','sum_standard','sum_staff','sum_inclusive','sum_other','sum_web_c','sum_mobile_app_c','sum_other_c','capacity','departurecountry','arrivalcountry','routetype','hour','avg_seat_res','max_seat_res','min_seat_res']]

y = df1['load_factor']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = False)




In [0]:
df1['past_seatssold'] = (df1['sum_business_reasonfortravel'] + df1['sum_leisure_reasonfortravel'] + df1['sum_other_reasonfortravel'])
df1['past_loadfactor'] = ((df1['past_seatssold']/df1['capacity']) * 100)


df1['alphalf'] = (                                                               # past data for load factor based on route
    df1.groupby('alpha_route')['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['alphalf'] = df1['alphalf'].fillna(df1['load_factor'].mean())

conditions = [
    df1['month'].isin([1]),
    df1['month'].isin([2]),
    df1['month'].isin([3]),
    df1['month'].isin([4]),
    df1['month'].isin([5]),
    df1['month'].isin([6]),
    df1['month'].isin([7]),
    df1['month'].isin([8]),
    df1['month'].isin([9]),
    df1['month'].isin([10]),
    df1['month'].isin([11]),
    df1['month'].isin([12])
]
choices = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
                                                                                       
df1['Months'] = np.select(conditions, choices)# past data based on seasons

df1['monthlf'] = (
    df1.groupby('Months')['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['monthlf'] = df1['monthlf'].fillna(df1['load_factor'].mean())

df1['alphabymonthlf'] = (
    df1.groupby(['Months','alpha_route'])['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['alphabymonthlf'] = df1['alphabymonthlf'].fillna(df1['load_factor'].mean())

df1['alphabydaylf'] = (
    df1.groupby(['dayoftheweek','alpha_route'])['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['alphabydayhlf'] = df1['alphabydaylf'].fillna(df1['load_factor'].mean())

df1['dayoftheweek'] = df1['flight_date'].dt.dayofweek

df1['daylf'] = (
    df1.groupby('dayoftheweek')['load_factor']
       .transform(lambda x: x.expanding().mean().shift(1))
)
df1['daylf'] = df1['daylf'].fillna(df1['load_factor'].mean())

inc = ((df1['past_seatssold']/df1['seatssold']) * 100)
df1['inc_loadfactor'] = (100 - inc)

df1['increase_lf'] = (                                                              
    df1.groupby('alpha_route')['inc_loadfactor']
       .transform(lambda x: x.rolling(window=3).mean().shift(1))
)
df1['increase_lf'] = df1['increase_lf'].fillna(df1['inc_loadfactor'].mean())

df1['alpha_lag_lf'] = (                                                              
    df1.groupby('alpha_route')['load_factor']
       .transform(lambda x: x.rolling(window=3).mean().shift(1))
)
df1['alpha_lag_lf'] = df1['alpha_lag_lf'].fillna(df1['load_factor'].mean())


X = df1[['avg_ticket_price','min_ticket_price', 'max_ticket_price','avg_lead_time','total_flight_time','sum_standard','sum_staff','sum_inclusive','sum_other','sum_web_c','sum_mobile_app_c','sum_other_c','capacity','departurecountry','arrivalcountry','routetype','past_loadfactor','alphalf','monthlf','daylf','alphabymonthlf','hour','alphabydaylf','avg_seat_res','max_seat_res','min_seat_res','alpha_lag_lf','increase_lf']]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = False)


In [0]:


model = LGBMRegressor (metric='rmse', learning_rate=0.016620389950346058 , max_depth = 12, min_child_samples= 6, n_estimators= 1000 ,num_leaves= 200, reg_alpha=49.99999999999999, reg_lambda=0.001, colsample_bytree= 0.7260055472722484, subsample = 1.0 ,random_state=42, early_stopping_rounds=30)   ## add parameters
model.fit(X_train, y_train, eval_set=[(X_test, y_test)])


y_prediction = model.predict(X_test)

print("The score is: ",r2_score(y_test,y_prediction))
print("Mean squared error is ",mean_squared_error(y_test, y_prediction))
print("Mean absolute error is ",mean_absolute_error(y_test, y_prediction))
print("Root mean squared error:", np.sqrt(mean_squared_error(y_test, y_prediction)))




In [0]:
A319 = 0
A320 = 0
A320H = 0
A321 = 0

X_test['predictedss'] = ((y_prediction * X_test['capacity'])/100)
g = 0

X_test['new_capacity'] = X_test['capacity'].copy()
col_idx = X_test.columns.get_loc('new_capacity')

while g < len(X_test):
    if X_test.iloc[g]['predictedss'] <= 235:
        if X_test.iloc[g]['predictedss'] <= 186:
            if X_test.iloc[g]['predictedss'] <= 180:
                if X_test.iloc[g]['predictedss'] <= 156:
                    #A319
                    X_test.iloc[g, col_idx] = 156
                    new_expenses156 =(X_test.iloc[g]['total_flight_time'] /60 ) * 2500
                    A319 = A319 + new_expenses156
                    g = g+1
                else:
                    #A320
                    X_test.iloc[g, col_idx] = 180
                    new_expenses180 =(X_test.iloc[g]['total_flight_time'] /60 ) * 2900
                    A320 = A320 + new_expenses180
                    g = g+1
            else:
                #A320H
                X_test.iloc[g, col_idx] = 186
                new_expenses186 =(X_test.iloc[g]['total_flight_time'] /60 ) * 3000
                A320H = A320H + new_expenses186
                g = g+1
        else:
            #A321
            X_test.iloc[g, col_idx] = 235
            new_expenses235 =(X_test.iloc[g]['total_flight_time'] /60 ) * 3500
            A321 = A321 + new_expenses235
            g = g+1
    else:
        #A321
        X_test.iloc[g, col_idx] = 235
        new_expenses235 =(X_test.iloc[g]['total_flight_time'] /60 ) * 3500
        A321 = A321 + new_expenses235
        g = g+1

cost = {156: 2500, 180: 2900, 186: 3000, 235: 3500}
old_expenses = 0
h=0
while h < len(X_test):
    rate = cost.get(X_test.iloc[h]['capacity'], 3500)
    old_expenses = old_expenses + ((X_test.iloc[h]['total_flight_time'] /60 ) * rate)
    h=h+1

saves = old_expenses - A319 - A320 - A320H - A321


old_revenue = (X_test['avg_ticket_price'] * df1['seatssold']).sum()
new_revenue = (X_test['avg_ticket_price'] * X_test['predictedss']).sum()

old_profit = old_revenue - old_expenses
new_profit = new_revenue - (A319 + A320 + A320H + A321)


print("The old profit is: £", round(old_profit, 2))
print("The new profit is: £", round(new_profit, 2))
print("The profit increase is: £", round(new_profit - old_profit, 2))






In [0]:
%skip
import xgboost as xgb

Model = xgb.XGBRegressor(objective ='reg:squarederror', learning_rate = 0.05444213852579246, max_depth = 10, n_estimators = 233, reg_alpha = 20.0, reg_lambda = 3.158673818177567e-06)


Model.fit(X_train, y_train)

y_prediction = Model.predict(X_test)


print("The score is: ",r2_score(y_test,y_prediction))
print("Mean squared error is ",mean_squared_error(y_test, y_prediction))
print("Mean absolute error is ",mean_absolute_error(y_test, y_prediction))
print("Root mean squared error:", np.sqrt(mean_squared_error(y_test, y_prediction)))




In [0]:
%skip
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler




A = df1[test_columns]
B = df1[target]

train1, test1, train2 , test2 = train_test_split (A, B, test_size=0.2, shuffle = False)






scaler = StandardScaler()
train1 = scaler.fit_transform(train1)
train2 = train2
test1 = scaler.transform(test1)
test2 = test2

mlp = MLPRegressor(random_state=42, max_iter=500, early_stopping = True, n_iter_no_change= 3)
mlp.fit(train1, train2)

y_prediction = mlp.predict(test1)

print("The score is: ",r2_score(test2,y_prediction))
print("Mean squared error is ",mean_squared_error(test2, y_prediction))
print("Mean absolute error is ",mean_absolute_error(test2, y_prediction))
print("Root mean squared error:", np.sqrt(mean_squared_error(test2, y_prediction)))

In [0]:
%skip
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
   ## add parameters

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    
    
}

grid = GridSearchCV(LGBMRegressor(), param_grid, cv = TimeSeriesSplit(n_splits=5), scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)






In [0]:
%skip
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import loguniform, uniform, randint

param_dist: dict [str, any]= {
    'reg_alpha': loguniform(1e-8,50),
    'reg_lambda': loguniform(1e-8,50),
    'learning_rate': loguniform(1e-3,3e-1),
    'max_depth': randint(3,20),
    'n_estimators': randint(10,500),
    'min_child_samples': randint(5,50),
}

random = RandomizedSearchCV(
    LGBMRegressor(),
    param_dist,
    n_iter=25,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_squared_error'
)

random.fit(X_train, y_train)

print("Best parameters:", random.best_params_)

In [0]:
%skip
import mlflow
mlflow.autolog(disable=True)

from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb

param_space = {
    'reg_alpha': Real(1e-2, 50, prior='log-uniform'),
    'reg_lambda': Real(1e-3, 50, prior='log-uniform'),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'max_depth': Integer(4, 15),
    'num_leaves': Integer(127, 200),
    'min_child_samples': Integer(5, 50),
    'colsample_bytree': Real(0.5, 1.0),
    'subsample': Real(0.5, 1.0),
}


bayes = BayesSearchCV(
    LGBMRegressor(n_estimators=1000, random_state=42, verbose=-1,early_stopping_rounds=30),
    param_space,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_squared_error',
    n_iter=25,
    verbose=0
)

bayes.fit(X_train, y_train, eval_set=[(X_test, y_test)])

print("Best parameters:", bayes.best_params_)

y_prediction = bayes.best_estimator_.predict(X_test)
print("R² score:", r2_score(y_test, y_prediction))
print("Mean squared error:", mean_squared_error(y_test, y_prediction))
print("Mean absolute error:", mean_absolute_error(y_test, y_prediction))
print("Root mean squared error:", np.sqrt(mean_squared_error(y_test, y_prediction)))
